Amishi Gupta

23/CS/048

In [1]:
!wget https://ocw.mit.edu/ans7870/6/6.006/s08/lecturenotes/files/t8.shakespeare.txt

--2026-04-13 06:45:24--  https://ocw.mit.edu/ans7870/6/6.006/s08/lecturenotes/files/t8.shakespeare.txt
Resolving ocw.mit.edu (ocw.mit.edu)... 151.101.130.132, 151.101.66.132, 151.101.2.132, ...
Connecting to ocw.mit.edu (ocw.mit.edu)|151.101.130.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5458199 (5.2M) [text/plain]
Saving to: ‘t8.shakespeare.txt’

t8.shakespeare.txt  100%[===================>]   5.21M   443KB/s    in 24s     

2026-04-13 06:45:51 (218 KB/s) - ‘t8.shakespeare.txt’ saved [5458199/5458199]



In [2]:
import torch
import torch.nn as nn
import math

In [3]:
with open("t8.shakespeare.txt", "r") as f:
    text = f.read()

#Character-level tokenization
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

#train/val split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0)/d_model))

        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [5]:
def attention(q, k, v, mask=None):
    d_k = q.size(-1)
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return weights @ v

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.d_model = d_model
        self.heads = heads
        self.d_k = d_model // heads

        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.fc = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, C = x.shape

        q = self.q(x).view(B, T, self.heads, self.d_k).transpose(1,2)
        k = self.k(x).view(B, T, self.heads, self.d_k).transpose(1,2)
        v = self.v(x).view(B, T, self.heads, self.d_k).transpose(1,2)

        out = attention(q, k, v, mask)
        out = out.transpose(1,2).contiguous().view(B, T, C)
        return self.fc(out)

In [7]:
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.ReLU(),
            nn.Linear(4*d_model, d_model)
        )

    def forward(self, x):
        return self.net(x)

In [8]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, heads)
        self.ff = FeedForward(d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x

In [9]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, heads)
        self.ff = FeedForward(d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask):
        x = x + self.self_attn(self.norm1(x), mask)
        x = x + self.ff(self.norm2(x))
        return x

In [12]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, heads=4, layers=2):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)
        self.blocks = nn.ModuleList([
            DecoderBlock(d_model, heads) for _ in range(layers)
        ])

        self.ln = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.shape

        mask = torch.tril(torch.ones(T, T)).to(x.device)

        x = self.embed(x)
        x = self.pos(x)

        for block in self.blocks:
            x = block(x, mask)

        x = self.ln(x)
        return self.fc(x)

In [13]:
model = Transformer(vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
def get_batch(data, block_size=64, batch_size=32):
    ix = torch.randint(len(data)-block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

for step in range(2000):
    xb, yb = get_batch(train_data)

    logits = model(xb)
    loss = loss_fn(logits.view(-1, vocab_size), yb.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 200 == 0:
        print(step, loss.item())

0 4.793764114379883
200 2.220696449279785
400 2.032097339630127
600 1.8220301866531372
800 1.8076304197311401
1000 1.714103102684021
1200 1.6978340148925781
1400 1.5106773376464844
1600 1.5954293012619019
1800 1.6116024255752563


In [14]:
def generate(model, start, max_new_tokens=200):
    model.eval()
    x = torch.tensor([encode(start)], dtype=torch.long)
    for _ in range(max_new_tokens):
        logits = model(x)
        probs = torch.softmax(logits[:, -1, :], dim=-1)
        next_token = torch.multinomial(probs, 1)
        x = torch.cat([x, next_token], dim=1)
    return decode(x[0].tolist())
print(generate(model, "ROMEO:"))

ROMEO:
   BotDEMONA. One that gone preset your spoke?                                        Er                       I      
          CHEver                                Exche                         My
